In [14]:
import pandas as pd
import numpy as np
import pickle
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [15]:
print("STEP 1: Loading dataset...")
fake_df = pd.read_csv("Fake.csv")
real_df = pd.read_csv("True.csv")

fake_df["label"] = 1
real_df["label"] = 0

df = pd.concat([fake_df, real_df], ignore_index=True)
df = df[["text", "label"]].dropna()
df = df[df["text"].str.strip() != ""]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"  Total rows   : {len(df)}")
print(f"  True News (0): {df[df['label']==0].shape[0]}")
print(f"  Rumor     (1): {df[df['label']==1].shape[0]}")


STEP 1: Loading dataset...
  Total rows   : 44267
  True News (0): 21416
  Rumor     (1): 22851


In [16]:
print("\nSTEP 2: Cleaning text...")
english_stopwords = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    punctuation_to_remove = string.punctuation.replace("-", "")
    text = text.translate(str.maketrans("", "", punctuation_to_remove))
    words = word_tokenize(text)
    preserved_words = {"5g", "covid-19", "nasa", "rover", "perseverance"}
    filtered_words = [
        word for word in words
        if word not in english_stopwords or word in preserved_words
    ]
    return " ".join(filtered_words)

df['cleaned_text'] = df['text'].fillna("").apply(clean_text)
print("  Done.")


STEP 2: Cleaning text...
  Done.


In [17]:
print("\nSTEP 3: Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f"  Training samples: {len(X_train)}")
print(f"  Testing  samples: {len(X_test)}")


STEP 3: Splitting data...
  Training samples: 35413
  Testing  samples: 8854


In [18]:
print("\nSTEP 4: Vectorizing...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print(f"  Feature matrix shape: {X_train_tfidf.shape}")


STEP 4: Vectorizing...
  Feature matrix shape: (35413, 5000)


In [19]:
print("\nSTEP 5: Training model...")
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train_tfidf, y_train)
print("  Done.")


STEP 5: Training model...
  Done.


In [20]:
print("\nSTEP 6: Evaluating...")
y_pred = model.predict(X_test_tfidf)
print(f"\n  Accuracy: {round(accuracy_score(y_test, y_pred)*100, 2)}%")
print()
print(classification_report(y_test, y_pred,
      target_names=["True News (0)", "Rumor (1)"]))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"                  Predicted True  Predicted Rumor")
print(f"  Actual True     {cm[0][0]:<16} {cm[0][1]}")
print(f"  Actual Rumor    {cm[1][0]:<16} {cm[1][1]}")


STEP 6: Evaluating...

  Accuracy: 98.89%

               precision    recall  f1-score   support

True News (0)       0.99      0.99      0.99      4283
    Rumor (1)       0.99      0.99      0.99      4571

     accuracy                           0.99      8854
    macro avg       0.99      0.99      0.99      8854
 weighted avg       0.99      0.99      0.99      8854

Confusion Matrix:
                  Predicted True  Predicted Rumor
  Actual True     4240             43
  Actual Rumor    55               4516


In [21]:
print("\nSTEP 7: Saving model...")
with open("logistic_regression_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("  logistic_regression_model.pkl saved")
print("  tfidf_vectorizer.pkl saved")


STEP 7: Saving model...
  logistic_regression_model.pkl saved
  tfidf_vectorizer.pkl saved


In [23]:
# ── STEP 8: Sanity Check (REAL samples from dataset) ────────
print("\n" + "-"*55)
print("STEP 8: SANITY CHECK — Real samples from dataset")
print("-"*55)

RUMOR_THRESHOLD = 0.80

# Take real samples directly from training data
real_samples  = df[df["label"] == 0]["text"].sample(5, random_state=42).tolist()
rumor_samples = df[df["label"] == 1]["text"].sample(5, random_state=42).tolist()

test_cases = (
    [(text, "True News") for text in real_samples] +
    [(text, "Rumor")     for text in rumor_samples]
)

passed = 0
for text, expected in test_cases:
    cleaned    = clean_text(text)
    vec        = vectorizer.transform([cleaned])
    proba      = model.predict_proba(vec)[0]
    rumor_prob = proba[1]
    conf       = round(max(proba) * 100, 1)
    result     = "Rumor" if rumor_prob >= RUMOR_THRESHOLD else "True News"
    status     = "✅ PASS" if result == expected else "❌ FAIL"
    if result == expected:
        passed += 1
    print(f"{status} [{conf}%] {text[:60]}...")
    print(f"       Expected: {expected} | Got: {result}")
    print()

print(f"Result: {passed}/{len(test_cases)} passed")
if passed >= 8:
    print("✅ Model is ready! Replace app.py and restart Flask.")
else:
    print("⚠️  Paste output here for further help.")


-------------------------------------------------------
STEP 8: SANITY CHECK — Real samples from dataset
-------------------------------------------------------
✅ PASS [95.3%] NEW YORK (Reuters) - Donald Trump’s spokesman on Thursday re...
       Expected: True News | Got: True News

✅ PASS [91.8%] WASHINGTON (Reuters) - There is “simply no place” in America...
       Expected: True News | Got: True News

✅ PASS [98.1%] PARIS (Reuters) - French President Emmanuel Macron on Thursd...
       Expected: True News | Got: True News

✅ PASS [99.4%] ISTANBUL (Reuters) - Sixty people including a former militar...
       Expected: True News | Got: True News

✅ PASS [99.3%] WASHINGTON (Reuters) - Leaders in the U.S. Congress on Monda...
       Expected: True News | Got: True News

❌ FAIL [78.4%] There are no surprises with the results on the Republican si...
       Expected: Rumor | Got: True News

✅ PASS [97.7%] Ever since the passing of Supreme Court Justice Antonin Scal...
       Expected: Ru

In [24]:
import pickle

# Resave the model with current sklearn version
with open("logistic_regression_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("✅ Model resaved with current sklearn version!")

# Confirm it works
test = vectorizer.transform([clean_text("test news article")])
proba = model.predict_proba(test)
print("✅ predict_proba works:", proba)

✅ Model resaved with current sklearn version!
✅ predict_proba works: [[0.06555714 0.93444286]]


In [25]:
import sklearn
print("Notebook sklearn:", sklearn.__version__)

Notebook sklearn: 1.8.0
